In [0]:
import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)
from delta.tables import DeltaTable

# Runner parameters
dbutils.widgets.dropdown(
    "load_type", "incremental", ["full", "incremental"]
)
dbutils.widgets.text("processing_date", "")
dbutils.widgets.text("source_folder", "")

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()
source_folder = dbutils.widgets.get("source_folder").strip()

runner_run_id = str(uuid.uuid4())
runner_start_time = datetime.now(timezone.utc)

log_table_name = "workspace.default.pipeline_execution_logs"

print("Starting Bronze pipeline...")
print("Load type:", load_type)
print("Processing date:", processing_date)

try:
    # Execute the complete Bronze notebook
    result = dbutils.notebook.run(
        "./01_bronze_pipeline_final",
        0,
        {
            "load_type": load_type,
            "processing_date": processing_date,
            "source_folder": source_folder
        }
    )

    print("Bronze pipeline completed successfully!")
    print("Notebook result:", result)

except Exception as error:
    print("Bronze pipeline failed:", str(error))

    # Explicit schema matching the execution log table
    log_schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("layer", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("processing_date", StringType(), True),
        StructField("source_folder", StringType(), True),
        StructField("start_timestamp", TimestampType(), True),
        StructField("end_timestamp", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("rows_read", LongType(), True),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("rows_quarantined", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    failure_df = spark.createDataFrame(
        [(
            runner_run_id,
            "bronze",
            "workspace.default.bronze_coin_market_raw",
            load_type,
            processing_date or None,
            source_folder or "default folder",
            runner_start_time,
            datetime.now(timezone.utc),
            "FAILED",
            0,
            0,
            0,
            0,
            str(error)[:4000]
        )],
        schema=log_schema
    )

    # The logging table was created during earlier steps
    log_delta = DeltaTable.forName(spark, log_table_name)

    (
        log_delta.alias("target")
        .merge(
            failure_df.alias("source"),
            "target.run_id = source.run_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Failure saved to execution logs.")

    # Keep the runner marked as failed
    raise

In [0]:
from pyspark.sql import functions as F

# Verify the saved Bronze records
bronze_df = spark.table(
    "workspace.default.bronze_coin_market_raw"
)

print("Total Bronze records:", bronze_df.count())

# Display the most recent Bronze execution logs
recent_logs = (
    spark.table("workspace.default.pipeline_execution_logs")
    .filter(F.col("layer") == "bronze")
    .orderBy(F.col("end_timestamp").desc())
    .select(
        "processing_date",
        "status",
        "rows_read",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "end_timestamp"
    )
    .limit(3)
)

display(recent_logs)